# Project Alpha: RL Bot v63p

## 📋 TODO & Roadmap
- **Data Policy:** Should we apply `handle_zeros_as_nan` globally to all matrices?
- **Feature Scaling:** Finalize the `get_agent_view` dual-view scaling logic.
- **Performance:** Move heavy logic (`analyze_4d_regime`) into `core` modular library.

<details>
<summary><b>📜 Version History (v59 - v63)</b></summary>

### v63
- Added **Metric Registry** (Strong-typed, dual-view).
- Fixed **Index Poisoning** bug (Dates in Ticker column).
- Unified **Perception** and **Execution** in Audit packs.
- Added 4 New Pillars: Return Autocorr, Range Position, OBV Divergence, Convexity.

### v62
- Fixed **Object Reference Mutation** bug in `compute_alpha_ensemble`.
- Added `PostBuildVerifier`, `DeepDiveDebugger`, and `ForensicExporter`.

### v61
- Verified all metrics with Excel parity.
- Refactored `AlphaEngine` into Orchestrator pattern.

### v60
- Converted notebook code to modular system.
- Added Volatility Regime plots.

### v59
- Implemented **Result Pattern** for error handling.
- Switched to logarithmic returns globally.
</details>

## I. Initialization & Environment

In [1]:
# 1. Setup & Path
%load_ext autoreload
%autoreload 2

import sys
import gc
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display, clear_output
import plotly.io as pio

def add_project_root_to_path():
    current = Path.cwd()
    for path in [current] + list(current.parents):
        if path.name == "notebooks_RLVR":
            sys.path.insert(0, str(path))
            return path
        candidate = path / "notebooks_RLVR"
        if candidate.exists():
            sys.path.insert(0, str(candidate))
            return candidate
    raise RuntimeError("Could not find notebooks_RLVR directory")

add_project_root_to_path()

# 2. Display Settings
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
pd.set_option("display.precision", 4)

# 3. Project Imports
from core.analyzer import create_walk_forward_analyzer, run_headless_simulation
from core.auditor import SystemAuditor as SA
from core.builder import ParallelFeatureBuilder, FeatureCubeStitcher
from core.contracts import FilterPack, EngineInput, MarketObservation
from core.engine import AlphaEngine, AlphaCache
from core.environment import DiscoveryEnv
from core.features import generate_features
from core.logic import AlphaLogic
from core.paths import OUTPUT_DIR
from core.quant import QuantUtils, TickerEngine
from core.settings import GLOBAL_SETTINGS
from core.utils import SystemUtils as SU
from strategy.registry import STRATEGY_REGISTRY

# 4. Load Environment Paths
DATA_DIR = Path(SU.load_env_from_root("DATA_DIR"))
DATA_PATH_OHLCV = Path(SU.load_env_from_root("DATA_PATH_OHLCV"))
DATA_PATH_INDICES = Path(SU.load_env_from_root("DATA_PATH_INDICES"))
print(f"✓ Paths Initialized.\nDATA Dir: {DATA_DIR}\nOHLCV: {DATA_PATH_OHLCV}\nIndices: {DATA_PATH_INDICES}")

NOTEBOOKS_RLVR_ROOT: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR

OUTPUT_DIR: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR\output

✓ Paths Initialized.
DATA Dir: c:\Users\ping\Files_win10\python\py311\stocks\data
OHLCV: c:\Users\ping\Files_win10\python\py311\stocks\data\df_OHLCV_stocks_etfs.parquet
Indices: c:\Users\ping\Files_win10\python\py311\stocks\data\df_indices.parquet


## II. Data Pipeline

In [2]:
# # === RELOADING FROM PARQUET ===
# features_df = pd.read_parquet("output/features_df.parquet")
# macro_df = pd.read_parquet("output/macro_df.parquet")
# df_close_wide = pd.read_parquet("output/df_close_wide.parquet")
# df_atrp_wide = pd.read_parquet("output/df_atrp_wide.parquet")
# df_trp_wide = pd.read_parquet("output/df_trp_wide.parquet")

# df_indices = pd.read_parquet(DATA_PATH_INDICES, engine="pyarrow")
# df_ohlcv = pd.read_parquet(DATA_PATH_OHLCV, engine="pyarrow")
# df_fed = pd.read_parquet(
#     DATA_DIR / "High_Yield_Spread_T10Y2Y_Spread.parquet", engine="pyarrow"
# )

In [3]:
print("⏳ Loading DataFrames... takes ~6 minutes")
df_indices = pd.read_parquet(DATA_PATH_INDICES, engine="pyarrow")
df_ohlcv = pd.read_parquet(DATA_PATH_OHLCV, engine="pyarrow")
df_fed = pd.read_parquet(
    DATA_DIR / "High_Yield_Spread_T10Y2Y_Spread.parquet", engine="pyarrow"
)

print("⚡ Generating Features...")
features_df, macro_df = generate_features(
    df_ohlcv=df_ohlcv,
    df_indices=df_indices,
    df_fed=df_fed,
    benchmark_ticker=GLOBAL_SETTINGS["benchmark_ticker"],
)

print("🚀 Unstacking Wide Matrices...")
df_close_wide = df_ohlcv["Adj Close"].unstack(level=0)
df_atrp_wide = features_df["ATRP"].unstack(level=0).reindex_like(df_close_wide)
df_trp_wide = features_df["TRP"].unstack(level=0).reindex_like(df_close_wide)

if GLOBAL_SETTINGS["handle_zeros_as_nan"]:
    df_close_wide = df_close_wide.replace(0, np.nan)

df_close_wide = df_close_wide.ffill(limit=GLOBAL_SETTINGS["max_data_gap_ffill"])
df_close_wide = df_close_wide.fillna(GLOBAL_SETTINGS["nan_price_replacement"])

print(
    f"✅ Pipeline Complete. Tickers: {len(df_close_wide.columns)}, Days: {len(df_close_wide)}"
)

⏳ Loading DataFrames... takes ~6 minutes
⚡ Generating Features...
⚡ Generating Decoupled Features (Benchmark: SPY)...
🚀 Unstacking Wide Matrices...
✅ Pipeline Complete. Tickers: 1576, Days: 8426


In [4]:
# === SAVE TO PERSISTENCE SANDBOX ===
_fn = OUTPUT_DIR / "features_df.parquet"
features_df.to_parquet(_fn, engine="pyarrow", index=True)
print(f"Saved: {_fn}")

_fn = OUTPUT_DIR / "macro_df.parquet"
macro_df.to_parquet(_fn, engine="pyarrow", index=True)
print(f"Saved: {_fn}")

_fn = OUTPUT_DIR / "df_close_wide.parquet"
df_close_wide.to_parquet(_fn, engine="pyarrow", index=True)
print(f"Saved: {_fn}")

_fn = OUTPUT_DIR / "df_atrp_wide.parquet"
df_atrp_wide.to_parquet(_fn, engine="pyarrow", index=True)
print(f"Saved: {_fn}")

_fn = OUTPUT_DIR / "df_trp_wide.parquet"
df_trp_wide.to_parquet(_fn, engine="pyarrow", index=True)
print(f"Saved: {_fn}")

Saved: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR\output\features_df.parquet
Saved: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR\output\macro_df.parquet
Saved: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR\output\df_close_wide.parquet
Saved: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR\output\df_atrp_wide.parquet
Saved: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR\output\df_trp_wide.parquet


## III. The Analysis Suite

In [5]:
# SINGLE SOURCE OF TRUTH
engine = AlphaEngine(
    df_ohlcv=df_ohlcv,
    features_df=features_df,
    macro_df=macro_df,
    df_close_wide=df_close_wide,
    df_atrp_wide=df_atrp_wide,
    df_trp_wide=df_trp_wide,
)
print("🎯 Master AlphaEngine Ready.")

🎯 Master AlphaEngine Ready.


In [6]:
# Configuration for Interactive UI
_inputs = EngineInput(
    mode="Ranking",
    decision_date=pd.Timestamp("2026-04-16"),
    lookback_period=189,
    holding_period=5,
    metric="Sharpe (TRP)",
    benchmark_ticker=GLOBAL_SETTINGS["benchmark_ticker"],
    rank_start=1,
    rank_end=200,
    debug=False,
)

analyzer1, stage1_pack = create_walk_forward_analyzer(
    engine, _inputs, universe_subset=None
)
analyzer1.show()

DEBUG: 938 stocks passed filters on 2026-04-16


In [7]:
print("--- CASCADE FILTER, TICKERS PASSED FIRST FILTER ---")
# Get decision date from last run
decision_date_last_run = FilterPack(decision_date=analyzer1.last_run.decision_date)

# 1. LAUNCH STAGE 2 (Cascade)
# universe_subset=analyzer1.last_run.tickers means "Scan the whole market"
analyzer2, stage2_pack = create_walk_forward_analyzer(
    engine,
    universe_subset=analyzer1.last_run.tickers,
    # universe_subset=None,
    filter_pack=decision_date_last_run,
)

print("🚀 Ready for Stage 2: Run Simulation for 2nd filter.")
analyzer2.show()

--- CASCADE FILTER, TICKERS PASSED FIRST FILTER ---
🚀 Ready for Stage 2: Run Simulation for 2nd filter.


In [8]:
# Configuration for Interactive UI
_inputs = EngineInput(
    mode="Ranking",
    decision_date=pd.Timestamp("2026-04-16"),
    lookback_period=10,
    holding_period=5,
    metric="Log Price Gain",
    benchmark_ticker=GLOBAL_SETTINGS["benchmark_ticker"],
    rank_start=1,
    rank_end=3,
    debug=False,
)

print("--- 🤖 RL AGENT HEADLESS VIEW ---")
metrics_df = run_headless_simulation(engine, _inputs)
display(metrics_df)
print(f"\nTarget Reward Signal: {metrics_df.loc['Group Gain', 'Holding']:.6f}")

--- 🤖 RL AGENT HEADLESS VIEW ---
DEBUG: 938 stocks passed filters on 2026-04-16
----------------------------------------------------------------------
Timeline: [2026-04-01] -> Decision: 2026-04-16 -> Entry: 2026-04-17 -> End: 2026-04-24
Selected Tickers (3):
SOXL, CRDO, NBIS
----------------------------------------------------------------------


,Full,Lookback,Holding
Metric,,,
Group Gain,0.6818,0.5047,0.1557
Benchmark Gain,0.0858,0.0684,0.0053
== Gain Delta,0.5960,0.4363,0.1504
Group Sharpe,23.1077,24.1712,38.7616
Benchmark Sharpe,10.9495,14.0962,2.3401
== Sharpe Delta,12.1582,10.0750,36.4215
Group Sharpe (ATRP),0.6272,0.6934,0.5230
Benchmark Sharpe (ATRP),0.3892,0.4683,0.0879
== Sharpe (ATRP) Delta,0.2380,0.2251,0.4351



Target Reward Signal: 0.155737


In [9]:
##################################
##################################
##################################

In [10]:
result = SU.visualize_analyzer_structure(analyzer=analyzer1)

🔍 HIGH-TRANSPARENCY AUDIT MAP
[  0] 📦 audit_pack (EngineOutput)
[  1]   📈 portfolio_series (shape=(196,))
[  2]   📈 benchmark_series (shape=(196,))
[  3]   🧮 normalized_plot_data (shape=(196, 200))
[  4]   📂 tickers (len=200)
[  5]     📄 index_0 (str)
[  6]     📄 index_1 (str)
[  7]     📄 index_2 (str)
[  8]     📄 index_3 (str)
[  9]     📄 index_4 (str)
[ 10]     📄 index_5 (str)
[ 11]     📄 index_6 (str)
[ 12]     📄 index_7 (str)
[ 13]     📄 index_8 (str)
[ 14]     📄 index_9 (str)
[ 15]     📄 index_10 (str)
[ 16]     📄 index_11 (str)
[ 17]     📄 index_12 (str)
[ 18]     📄 index_13 (str)
[ 19]     📄 index_14 (str)
[ 20]     📄 index_15 (str)
[ 21]     📄 index_16 (str)
[ 22]     📄 index_17 (str)
[ 23]     📄 index_18 (str)
[ 24]     📄 index_19 (str)
[ 25]     📄 index_20 (str)
[ 26]     📄 index_21 (str)
[ 27]     📄 index_22 (str)
[ 28]     📄 index_23 (str)
[ 29]     📄 index_24 (str)
[ 30]     📄 index_25 (str)
[ 31]     📄 index_26 (str)
[ 32]     📄 index_27 (str)
[ 33]     📄 index_28 (str)
[

In [11]:
target_tickers = SU.peek(4, result)

 📍 INDEX: [4]
 🏷️  NAME:  tickers
 📂 PATH:  audit_pack -> tickers



['SGOV',
 'SHV',
 'BIL',
 'MINT',
 'USFR',
 'BOXX',
 'PULS',
 'JPST',
 'GTLS',
 'SNDK',
 'JAAA',
 'RVMD',
 'CIEN',
 'TER',
 'EA',
 'LITE',
 'ASX',
 'VGSH',
 'WDC',
 'GLW',
 'ROIV',
 'SHY',
 'ROST',
 'EWY',
 'WBD',
 'VALE',
 'MU',
 'FTI',
 'ERIC',
 'RIO',
 'VTEB',
 'BE',
 'KEYS',
 'NOK',
 'STX',
 'FIX',
 'UTHR',
 'VCSH',
 'IGSB',
 'MUB',
 'CHRW',
 'EMXC',
 'LRCX',
 'GOOGL',
 'GOOG',
 'HSBC',
 'INTC',
 'TEVA',
 'JNJ',
 'TD',
 'CAT',
 'EWZ',
 'ALB',
 'MKSI',
 'CMI',
 'BNS',
 'BSV',
 'COPX',
 'RY',
 'SU',
 'PBR',
 'EFV',
 'ASML',
 'GSK',
 'EWT',
 'FDX',
 'ONTO',
 'BP',
 'AA',
 'IAG',
 'CVE',
 'SLV',
 'HL',
 'KGC',
 'ITUB',
 'COHR',
 'BHP',
 'AU',
 'JAZZ',
 'IAUM',
 'AMAT',
 'GLDM',
 'SGOL',
 'IAU',
 'APA',
 'B',
 'AZN',
 'XBI',
 'SCCO',
 'EMB',
 'PSLV',
 'EQX',
 'SOXX',
 'SCHF',
 'GLD',
 'VEA',
 'BG',
 'SPDW',
 'NEM',
 'IEMG',
 'VEU',
 'IONS',
 'GDX',
 'EEM',
 'VXUS',
 'PHYS',
 'LSCC',
 'HAL',
 'IXUS',
 'ETR',
 'EWJ',
 'MTZ',
 'OPEN',
 'SMH',
 'FNDX',
 'USO',
 'SOXL',
 'KLAC',
 'GDXJ',
 'M

In [12]:
SU.export_audit_to_excel(
    audit_pack=analyzer1.last_run, filename="Audit_Verification_Report.xlsx"
)

📂 [EXCEL AUDIT] Building full transparency report: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR\output\Audit_Verification_Report.xlsx
✨ Audit Report Complete: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR\output\Audit_Verification_Report.xlsx


WindowsPath('C:/Users/ping/Files_win10/python/py311/stocks/notebooks_RLVR/output/Audit_Verification_Report.xlsx')

In [13]:
##################################
##################################
##################################

In [14]:
import pathlib

# Set display width to prevent line wrapping
pd.set_option("display.width", 2000)

# Show all columns instead of truncated view
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)


def load_latest_finviz_parquet(data_dir: str) -> pd.DataFrame:
    path = pathlib.Path(data_dir)
    pattern = "[0-9][0-9][0-9][0-9]-[0-9][0-9]-[0-9][0-9]_df_finviz_merged_stocks_etfs.parquet"

    files = list(path.glob(pattern))

    if not files:
        print(f"DEBUG: No files matching pattern found in {path.absolute()}")
        return None

    latest_path = max(files, key=lambda x: x.name)
    print(f"DEBUG: Loading file: {latest_path.name}")

    try:
        return pd.read_parquet(latest_path)
    except Exception as e:
        print(f"DEBUG: Failed to read {latest_path.name}. Error: {e}")
        return None


# Usage
df_finviz = load_latest_finviz_parquet(DATA_DIR)
print(f"df_finviz:\n{df_finviz}")

DEBUG: Loading file: 2026-07-22_df_finviz_merged_stocks_etfs.parquet
df_finviz:
         No.                                            Company               Index                  Sector                                  Industry         Country Exchange                                               Info  MktCap AUM, M  Rank  Market Cap, M     P/E  Fwd P/E    PEG      P/S     P/B       P/C    P/FCF    Book/sh    Cash/sh  Dividend %  Dividend TTM Dividend Ex Date  Payout Ratio %       EPS  EPS next Q  EPS this Y %  EPS next Y %  EPS past 5Y %  EPS next 5Y %  Sales past 5Y %  Sales Q/Q %  EPS Q/Q %  EPS YoY TTM %  Sales YoY TTM %   Sales, M  Income, M  EPS Surprise %  Revenue Surprise %  Outstanding, M  Float, M  Float %  Insider Own %  Insider Trans %  Inst Own %  Inst Trans %  Short Float %  Short Ratio  Short Interest, M   ROA %     ROE %  ROIC %  Curr R  Quick R  LTDebt/Eq  Debt/Eq  Gross M %  Oper M %  Profit M %  Perf 3D %  Perf Week %  Perf Month %  Perf Quart %  Perf Half %  Perf

In [15]:
import ast

try:
    result = df_finviz.loc[target_tickers]
except KeyError as e:
    # Extract missing tickers from pandas error message
    error_msg = str(e)
    missing_str = error_msg.replace(" not in index", "").strip('"').strip("'")
    missing_tickers = ast.literal_eval(missing_str)

    print(f"Missing tickers: {missing_tickers}")

    # Remove missing tickers and retry
    target_tickers_in_finviz = [t for t in target_tickers if t not in missing_tickers]
    result = df_finviz.loc[target_tickers_in_finviz]

# print(f'missing_tickers: {missing_tickers}\n')
print(f"result:\n{result}")

Missing tickers: ['GTLS', 'OPEN', 'USO', 'SIL']
result:
        No.                                            Company               Index                  Sector                                  Industry         Country Exchange                                               Info  MktCap AUM, M  Rank  Market Cap, M     P/E  Fwd P/E   PEG      P/S     P/B      P/C   P/FCF  Book/sh  Cash/sh  Dividend %  Dividend TTM Dividend Ex Date  Payout Ratio %    EPS  EPS next Q  EPS this Y %  EPS next Y %  EPS past 5Y %  EPS next 5Y %  Sales past 5Y %  Sales Q/Q %  EPS Q/Q %  EPS YoY TTM %  Sales YoY TTM %   Sales, M  Income, M  EPS Surprise %  Revenue Surprise %  Outstanding, M  Float, M  Float %  Insider Own %  Insider Trans %  Inst Own %  Inst Trans %  Short Float %  Short Ratio  Short Interest, M  ROA %    ROE %  ROIC %  Curr R  Quick R  LTDebt/Eq  Debt/Eq  Gross M %  Oper M %  Profit M %  Perf 3D %  Perf Week %  Perf Month %  Perf Quart %  Perf Half %  Perf Year %  Perf YTD %  Beta     ATR  AT

In [16]:
def cleanup_ui():
    clear_output(wait=True)
    pio.renderers.default = None
    gc.collect()
    print("🧹 Memory Cleared.")


def active_engine_audit():
    gc.collect()
    engines = [obj for obj in gc.get_objects() if type(obj).__name__ == "AlphaEngine"]
    print(f"Active AlphaEngine Instances: {len(engines)}")


active_engine_audit()

Active AlphaEngine Instances: 1


In [17]:
# print(features_df.info())
# display(features_df.head())
# display(df_indices.tail())

In [18]:
################################
################################
################################
################################

In [19]:
SU.peek(0, result)

❌ Error: Pass the result map (list), not the analyzer object.


In [20]:
features_df.loc["TSLA"]

,ATR,ATRP,TRP,RSI,Mom_21,Consistency,IR_63,Beta_63,DD_21,AutoCorr_15,Ret_1d,Range_Pos_20,Slope_P_5,Slope_V_5,Convexity,RollingStalePct,RollMedDollarVol,RollingSameVolCount
Date,,,,,,,,,,,,,,,,,,
2010-06-29,NaN,0.0000,0.0000,50.0000,NaN,NaN,NaN,1.0000,0.0000,0.0000e+00,NaN,NaN,NaN,NaN,0.0000e+00,NaN,NaN,NaN
2010-06-30,0.4747,0.2988,0.2988,0.0000,NaN,NaN,NaN,1.0000,0.0000,0.0000e+00,-2.5115e-03,NaN,NaN,NaN,0.0000e+00,NaN,NaN,NaN
2010-07-01,0.4677,0.3194,0.2573,0.0000,NaN,NaN,NaN,1.0000,0.0000,0.0000e+00,-7.8474e-02,NaN,NaN,NaN,0.0000e+00,NaN,NaN,NaN
2010-07-02,0.4552,0.3556,0.2286,0.0000,NaN,NaN,NaN,1.0000,0.0000,0.0000e+00,-1.2568e-01,NaN,NaN,NaN,0.0000e+00,NaN,NaN,NaN
2010-07-06,0.4425,0.4120,0.2588,0.0000,NaN,0.0,NaN,1.0000,0.0000,0.0000e+00,-1.6094e-01,NaN,-1.0041e-01,-0.6060,0.0000e+00,NaN,NaN,NaN
2010-07-07,0.4188,0.3976,0.1044,0.0000,NaN,0.0,NaN,1.0000,0.0000,0.0000e+00,-1.9246e-02,NaN,-1.1317e-01,-0.5517,0.0000e+00,NaN,NaN,NaN
2010-07-08,0.3981,0.3420,0.1117,18.8807,NaN,0.2,NaN,1.0000,0.0000,0.0000e+00,1.0507e-01,NaN,-6.5352e-02,-0.3117,3.5056e-02,NaN,NaN,NaN
2010-07-09,0.3761,0.3242,0.0776,18.7429,NaN,0.2,NaN,1.0000,0.0000,0.0000e+00,-3.4364e-03,NaN,-1.1641e-02,-0.1776,1.0153e-01,NaN,NaN,NaN
2010-07-12,0.3544,0.3118,0.0628,17.9217,NaN,0.2,NaN,1.0000,0.0000,0.0000e+00,-2.0112e-02,NaN,2.0989e-02,-0.0842,8.6341e-02,NaN,NaN,NaN
